# Module 0 Worksheet — Tokens, Embeddings, Attention
Read `concept_notes.md` and `diagrams.md` in this folder first.

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../../wrapper_fix"))      # folder containing the corrected inhouse_wrappers.py
sys.path.append(os.path.abspath("../../inhouse_rag_capstone"))  # folder containing your real inhouse_llm.py

from inhouse_llm import MODEL_QWEN3_14B, MODEL_QWEN3_30B, MODEL_MISTRAL, MODEL_LLAMA, MODEL_DEVSTRAL, MODEL_QWEN2_5_VL_7B
from inhouse_wrappers import get_chat_model, InHouseEmbeddings, build_vision_messages, llm_for
from langchain_core.messages import SystemMessage, HumanMessage

embedder = InHouseEmbeddings()

def ask(system_prompt, user_prompt, model=MODEL_QWEN3_14B, max_tokens=500):
    """Correctly-routed replacement for calling multimodal_chat() directly."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]).content

def ask_vision(system_prompt, user_prompt, image_base64, model=MODEL_QWEN2_5_VL_7B, max_tokens=500):
    """Correctly-routed, correctly-formatted multimodal call."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke(build_vision_messages(system_prompt, user_prompt, image_base64)).content

print("Setup OK")

## 1. Tokenization in practice
`pip install tiktoken --break-system-packages` if missing. Note: this is an approximation of your in-house models' actual tokenizer, but close enough to build intuition for token-vs-word counts.

In [ ]:
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")

samples = [
    "What is RAG?",
    "unbelievable",
    "MCP standardizes tool calling.",
]
for s in samples:
    tokens = enc.encode(s)
    print(f"{s!r} -> {len(tokens)} tokens -> {[enc.decode([t]) for t in tokens]}")

## 2. Embedding lookup vs sentence embedding (the distinction in concept_notes.md section 1)
Jina gives ONE vector for a whole sentence — a pooled sentence embedding, not a per-token lookup. Confirm the shape.

In [ ]:
vec = embedder.embed_query("What is RAG?")
print("Vector length:", len(vec))
print("First 5 values:", vec[:5])

## 3. Manual self-attention on a toy example
Compute attention scores by hand with numpy on 3 toy "tokens" — this is the same Q/K/V mechanism from `diagrams.md` section 3, just at a scale you can fully see.

In [ ]:
import numpy as np

np.random.seed(0)
d = 4  # tiny embedding dim for illustration
tokens = ["What", "is", "RAG"]
X = np.random.randn(len(tokens), d)  # toy embeddings, NOT real model embeddings

Wq, Wk, Wv = (np.random.randn(d, d) for _ in range(3))
Q, K, V = X @ Wq, X @ Wk, X @ Wv

scores = Q @ K.T / np.sqrt(d)
weights = np.exp(scores) / np.exp(scores).sum(axis=1, keepdims=True)  # softmax
output = weights @ V

print("Attention weights (rows=query token, cols=key token):")
for i, t in enumerate(tokens):
    print(f"  {t:6s}: " + ", ".join(f"{tokens[j]}={weights[i,j]:.2f}" for j in range(len(tokens))))
print("\nEach row sums to 1.0 (softmax) — that row tells you how much\n"
      "that query token 'attends to' every other token.")

## Teaser exercise
Increase `d` to 64 and re-run. Does the qualitative pattern (which tokens attend most to which) change, or does it just get less interpretable by eye? This is exactly why real models need visualization tools to inspect attention — at real model scale (4096+ dims, dozens of heads) you can't eyeball it like this.